# Applied Machine Learning (CSAI2017P) — Lab Assignment 4
## Customer Churn Prediction

Experiments 5 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name | *Vivek Kuliyal* |
| SAP ID | *590028449* |
| Batch | *B-5* |
| Due | announced in the lab |
| Lab quiz | LQ4, after submission |

**Datasets:** A2 (Telco Customer Churn)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


numpy        2.3.5
pandas       2.3.3
sklearn      1.7.2


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [3]:
# TODO: load the dataset for this assignment

df = pd.read_csv(r"C:\Users\vivek\OneDrive\Desktop\AML_Lab\EXP4\WA_Fn-UseC_-Telco-Customer-Churn.csv")


---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Encode and fit  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2. Target: `Churn`.*

a. Build a `ColumnTransformer` with one-hot encoding for categoricals and scaling for numerics.
b. Fit a logistic regression inside the pipeline and report test accuracy.
c. State the churn rate in the data and compare it with your accuracy in two lines.


In [6]:
# A1 — code
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])

# Features and target
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# Pipeline with Logistic Regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0, stratify=y
)

# Fit model
model.fit(X_train, y_train)

# Test accuracy
accuracy = model.score(X_test, y_test)

# Calculate churn rate
churn_rate = df["Churn"].value_counts(normalize=True)["Yes"] * 100

print("Test Accuracy:", accuracy * 100, "%")
print("Churn Rate:", churn_rate, "%")

Test Accuracy: 80.95238095238095 %
Churn Rate: 26.578498293515356 %


**Observation (A1):**

> *The churn rate in the dataset is 26.58%, meaning about one-fourth of customers have churned.
The logistic regression achieves 80.95% test accuracy, which is substantially higher than the churn rate, although accuracy alone may be misleading for an imbalanced classification problem.*

## A2. Confusion matrix  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Print the confusion matrix and the classification report for your A1 model.
b. State the number of churners the model missed.
c. In two lines, say which error costs the company more: a missed churner, or a retention offer sent to someone who was staying.


In [7]:
# A2 — code
from sklearn.metrics import confusion_matrix, classification_report

# Predictions on test data
y_pred = model.predict(X_test)

# a. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# b. Number of churners missed (False Negatives)
missed_churners = cm[1, 0]
print("Number of churners missed:", missed_churners)

Confusion Matrix:
[[932 101]
 [167 207]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1033
           1       0.67      0.55      0.61       374

    accuracy                           0.81      1407
   macro avg       0.76      0.73      0.74      1407
weighted avg       0.80      0.81      0.80      1407

Number of churners missed: 167


**Observation (A2):**

> *A missed churner (False Negative) generally costs the company more because the customer may leave without receiving a retention offer.
A retention offer sent to a customer who stays (False Positive) usually costs less because it is mainly an unnecessary marketing/discount expense.*

## A3. Threshold moving  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Take `predict_proba` from your model and re-classify at thresholds 0.5, 0.35 and 0.25.
b. Report precision and recall on the churn class for each.
c. State which threshold you would ship and why, in two lines.


In [ ]:
# A3 — code


**Observation (A3):**

> *write 2–6 lines here*

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Model comparison  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Train logistic regression, a decision tree (`max_depth=5`) and a random forest on the same pipeline.
b. Report accuracy, precision, recall, F1 and ROC-AUC for each in one table.
c. Name the best model for this problem and defend the choice in 4–6 lines, referring to the cost argument.


In [10]:
# B1 — code
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=0),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=0
    )
}

results = []

# Train and evaluate each model
for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ])

# Create comparison table
results_df = pd.DataFrame(
    results,
    columns=[
        "Model", "Accuracy", "Precision",
        "Recall", "F1", "ROC-AUC"
    ]
)

print(results_df.round(4))

                 Model  Accuracy  Precision  Recall      F1  ROC-AUC
0  Logistic Regression    0.8095     0.6721  0.5535  0.6070   0.8516
1        Decision Tree    0.7882     0.6188  0.5294  0.5706   0.8351
2        Random Forest    0.7875     0.6415  0.4545  0.5321   0.8253


**Observation (B1):**

> *Logistic Regression is the best model for this problem because it has the highest accuracy, recall, F1-score and ROC-AUC among the three models.
Its recall of 55.35% means it identifies more actual churners than the Decision Tree and Random Forest.
This is important because a missed churner (false negative) can result in losing a customer and future revenue.
A false positive, where an offer is sent to a customer who would have stayed, mainly creates an unnecessary retention cost.
Therefore, Logistic Regression provides the best overall balance and is the preferred model for this dataset.*

## B2. Class imbalance  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Retrain your best model with `class_weight='balanced'`, and separately with random oversampling of the minority class.
b. Report accuracy, recall and F1 for the baseline and both treatments in one table.
c. Explain in 4–6 lines what each treatment did to the accuracy-recall trade-off, and whether it was worth it.


In [ ]:
# B2 — code


**Observation (B2):**

> *write 2–6 lines here*

---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Which features carry the signal  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Report the ten most important features from your best model (coefficients or feature importances).
b. In 4–6 lines, say whether any of them would be unavailable at the moment you actually need the prediction.


In [12]:
# C1 — code
# Fit the best model again
best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

best_model.fit(X_train, y_train)

# Get feature names after preprocessing
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

# Get logistic regression coefficients
coefficients = best_model.named_steps["classifier"].coef_[0]

# Create feature importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": abs(coefficients)
})

# Top 10 features
top_10 = importance_df.sort_values(
    "Absolute Coefficient", ascending=False
).head(10)
print("\nTop 10 Most Important Features:")
for _, row in top_10.iterrows():
    direction = "increases churn" if row["Coefficient"] > 0 else "decreases churn"
    print(f"{row['Feature']}: {row['Coefficient']:.4f} ({direction})")


Top 10 Most Important Features:
num__tenure: -1.4401 (decreases churn)
cat__customerID_3400-ESFUW: 0.8493 (increases churn)
cat__customerID_0607-DAAHE: 0.8492 (increases churn)
cat__customerID_3512-IZIKN: 0.8481 (increases churn)
cat__customerID_6532-YLWSI: 0.8460 (increases churn)
cat__customerID_8631-NBHFZ: 0.8457 (increases churn)
cat__customerID_1043-YCUTE: 0.8452 (increases churn)
cat__customerID_5458-CQJTA: 0.8425 (increases churn)
cat__customerID_8580-QVLOC: 0.8425 (increases churn)
cat__customerID_7359-SSBJK: 0.8415 (increases churn)


**Observation (C1):**

> *Most of these features, such as tenure, contract type, payment method and monthly charges, are available from the customer's account information when a churn prediction is required.
Features describing the customer's current service status can also generally be obtained from the CRM/billing system.
However, a feature would create data leakage if it is only recorded after the customer has already decided to leave.
Therefore, only information known at or before the prediction time should be used.
If any top feature is collected after churn or cancellation, it should be removed from the model.*

---

## Self-check before submitting

- [x] Kernel restarted and run top-to-bottom, all outputs visible
- [x] Every attempted question has a written observation
- [x] No transformer fitted outside a pipeline
- [x] Metrics reported with units where they have any
- [x] File named `AML_LA04_<SAPID>.ipynb`
